# Repetições individuais — PlantVillage
Selecione **um modelo e uma semente por execução**. Usa os scripts existentes, batch 32,
até 30 épocas, Adam, early stopping patience 5/min_delta 1e-4 e o mesmo CSV de splits.
A seed muda inicialização, augmentations e ordem de treino; não refaz a divisão.
A seed 42 permanece nos diretórios originais. As novas execuções ficam em
`results/repetitions/seed_<semente>/<modelo>/`. Diretórios existentes são bloqueados.
Apenas treino e validação. Nenhuma célula avalia imagens de teste.

No Colab: selecione runtime com GPU L4, monte o Drive abaixo e atualize **src/**
com esta versão do projeto. Use a pasta que contém os resultados L4 originais da seed 42.
O notebook não instala nem atualiza dependências automaticamente: mantenha as versões
PyTorch, torchvision e CUDA registradas na referência. Não execute os notebooks de split.


In [ ]:
from pathlib import Path
import sys
try:
    from google.colab import drive
except ImportError:
    IN_COLAB = False
else:
    IN_COLAB = True
    drive.mount('/content/drive')

PROJECT_ROOT = (Path('/content/drive/MyDrive/plant-disease-classification')
                if IN_COLAB else Path.cwd())
if not (PROJECT_ROOT / 'src').is_dir() and (PROJECT_ROOT.parent / 'src').is_dir():
    PROJECT_ROOT = PROJECT_ROOT.parent
IMAGE_ROOT = Path('/content/plantvillage_color') if IN_COLAB else PROJECT_ROOT / 'data/plantvillage_color'
ZIP_PATH = PROJECT_ROOT / 'data/plantvillage.zip'  # ajuste para o ZIP já usado no projeto
sys.path.insert(0, str(PROJECT_ROOT / 'src'))


In [ ]:
MODEL = 'baseline_cnn'  # baseline_cnn, resnet50, densenet201, efficientnet_b0
SEED = 123  # 123 ou 456
EXECUTE_TRAINING = False  # altere para True somente quando desejar iniciar


## Imagens locais (opcional)
Se as imagens já estão extraídas, ajuste IMAGE_ROOT e pule a próxima célula.
Caso contrário, ela extrai do ZIP **somente os caminhos de treino e validação**
do CSV existente. Não baixa dados, não gera splits e não abre imagens de teste.


In [ ]:
EXTRACT_IMAGES = False  # True somente se precisar extrair do ZIP
if EXTRACT_IMAGES:
    import pandas as pd
    import shutil
    import zipfile
    from plantvillage_pytorch import _raw_color_relative_parts
    metadata = pd.read_csv(PROJECT_ROOT / 'results/plantvillage_metadata_split.csv')
    selected = metadata.loc[metadata['split'].isin(['train', 'validation']), 'zip_path']
    with zipfile.ZipFile(ZIP_PATH) as archive:
        for member in selected:
            parts = _raw_color_relative_parts(member)
            target = IMAGE_ROOT.joinpath(*parts)
            if not target.resolve().is_relative_to(IMAGE_ROOT.resolve()):
                raise ValueError('Caminho inválido')
            if not target.exists():
                target.parent.mkdir(parents=True, exist_ok=True)
                with archive.open(member) as source, target.open('wb') as destination:
                    shutil.copyfileobj(source, destination)
    print('Extração de treino e validação concluída.')


In [ ]:
import json
from run_repetition import prepare_repetition, run_repetition
command, manifest = prepare_repetition(PROJECT_ROOT, MODEL, SEED, IMAGE_ROOT)
print(json.dumps(manifest, indent=2, ensure_ascii=False))


In [ ]:
if EXECUTE_TRAINING:
    output = run_repetition(PROJECT_ROOT, MODEL, SEED, IMAGE_ROOT)
    print('Execução concluída:', output)
else:
    print('Somente preparação. Para treinar, defina EXECUTE_TRAINING = True e execute esta célula.')


## Comparação de todas as execuções disponíveis
Lê apenas artefatos salvos. Não treina e não carrega checkpoints.
A seed 42 é lida do diretório original. As seeds 123 e 456 são lidas dos novos diretórios.
O desvio padrão é **amostral (ddof=1)** entre sementes; com uma execução fica indefinido (NaN).
`n_runs` mostra a quantidade válida: a comparação só fica completa com 3 seeds por modelo.
Resultados ausentes/inválidos não entram nas médias. Avisos de hardware/ambiente devem
ser resolvidos antes de interpretar tempos como uma comparação controlada.


In [ ]:
from compare_repetitions import compare_repetitions
comparison = compare_repetitions(PROJECT_ROOT / 'results')
display(comparison.individual)
display(comparison.aggregate)
for warning in comparison.warnings:
    print(warning)
